# HotPotQA LLM Question Answering — Best RAG Pipeline

Clean Colab notebook for the best-performing pipeline:

1. Load HotPotQA dev data from Google Drive  
2. Load Llama-3.1-8B via Unsloth  
3. Retrieve relevant evidence using sentence embeddings + FAISS  
4. Generate short `FINAL:` answers  
5. Save predictions with autosave/resume  
6. Wrap predictions for HotPotQA evaluator and compute EM/F1


In [ ]:
!nvidia-smi


In [ ]:
!pip -q install "unsloth[colab-new]"  # Unsloth Colab install
!pip -q install transformers accelerate


In [ ]:
!pip -q install sentence-transformers faiss-cpu

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, json

DATA_PATH = "/content/drive/MyDrive/hotpot/hotpot_dev_fullwiki_v1.json"
OUT_PATH  = "/content/drive/MyDrive/hotpot/pred_dev_rag1.json"

# (optional) make sure folder exists
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)

assert os.path.exists(DATA_PATH), f"Dataset not found at: {DATA_PATH}"
print("Dataset path OK:", DATA_PATH)
print("Output will be saved to:", OUT_PATH)


In [ ]:
import json


with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Examples:", len(data))
print("Keys:", data[0].keys())
print("First question:", data[0]["question"])

In [ ]:
from unsloth import FastLanguageModel
import torch

model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)


In [ ]:
import re
import torch

def build_prompt(question: str, evidence: str) -> str:
    return (
        "You are a QA system.\n"
        "Use ONLY the evidence below.\n"
        "Return EXACTLY one line in this format:\n"
        "FINAL: <answer>\n"
        "No explanations. No parentheses.\n\n"
        f"Evidence:\n{evidence}\n\n"
        f"Question: {question}\n"
        "FINAL: "   # NOTE the space after colon
    )

def clean_answer(ans: str) -> str:
    ans = ans.strip()
    ans = ans.replace("}|{span}", "").strip()
    ans = re.sub(r"_+", " ", ans).strip()
    ans = re.sub(r"\s+", " ", ans).strip()
    ans = re.split(r"\s*[\(\[\{]", ans, maxsplit=1)[0].strip()
    ans = ans.strip(" \t\r\n\"'`.,;: ")
    return ans

@torch.inference_mode()
def answer_batch(questions, evidences, max_new_tokens=16):
    prompts = [build_prompt(q, e) for q, e in zip(questions, evidences)]
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=1800,
    ).to(model.device)

    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        min_new_tokens=1,    # helps avoid empty output after FINAL:
        do_sample=False,
        temperature=0.0,
        top_p=1.0,
    )

    texts = tokenizer.batch_decode(out, skip_special_tokens=True)

    answers = []
    for t in texts:
        raw = t.rsplit("FINAL:", 1)[-1] if "FINAL:" in t else t.rsplit("Answer:", 1)[-1]
        raw = raw.strip()
        lines = raw.splitlines()
        first_line = lines[0].strip() if len(lines) > 0 else ""
        answers.append(clean_answer(first_line))

    return answers

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

# Small + fast embedder (good default)
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")

In [ ]:
import re

def _flatten_context(context, max_titles=12, max_sents_per_title=8, max_candidates=120):
    """
    context: [ [title, [sent1, sent2, ...]], ... ]
    Returns list of strings like: "Title: sentence"
    Caps candidates to keep embedding fast.
    """
    cands = []
    for title, sents in context[:max_titles]:
        for s in sents[:max_sents_per_title]:
            s = (s or "").strip()
            if not s:
                continue
            cands.append(f"{title}: {s}")
            if len(cands) >= max_candidates:
                return cands
    return cands

def rag_evidence(question, context, top_k=8, max_chars=4000,
                 max_titles=12, max_sents_per_title=8, max_candidates=120):
    cands = _flatten_context(context,
                             max_titles=max_titles,
                             max_sents_per_title=max_sents_per_title,
                             max_candidates=max_candidates)
    if not cands:
        return ""

    # Embed candidates and the question (cosine similarity via normalized vectors)
    cand_vecs = embedder.encode(cands, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)
    q_vec = embedder.encode([question], convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)

    dim = cand_vecs.shape[1]
    index = faiss.IndexFlatIP(dim)  # inner product on normalized vectors = cosine similarity
    index.add(cand_vecs.astype(np.float32))

    k = min(top_k, len(cands))
    scores, idxs = index.search(q_vec.astype(np.float32), k)

    picked = [cands[i] for i in idxs[0] if 0 <= i < len(cands)]
    evidence = "\n".join(f"- {p}" for p in picked)
    return evidence[:max_chars]

In [ ]:
import json, os
from tqdm import tqdm

BATCH_SIZE = 8
SAVE_EVERY = 200
MAX_NEW_TOKENS = 16

preds = {}
existing_ids = set()

# Resume if file already exists
if os.path.exists(OUT_PATH):
    with open(OUT_PATH, "r", encoding="utf-8") as f:
        preds = json.load(f)
    existing_ids = set(preds.keys())
    print("Resuming from existing file. Already done:", len(existing_ids))
new_done = 0

buffer_q, buffer_e, buffer_ids = [], [], []

for ex in tqdm(data):
    qid = ex["_id"]
    if qid in existing_ids:
        continue

    q = ex["question"]

    e = rag_evidence(q, ex["context"])

    buffer_ids.append(qid)
    buffer_q.append(q)
    buffer_e.append(e)

    if len(buffer_q) == BATCH_SIZE:
        answers = answer_batch(buffer_q, buffer_e, max_new_tokens=MAX_NEW_TOKENS)
        for i, a in zip(buffer_ids, answers):
            preds[i] = a

        new_done += len(buffer_ids)
        if new_done >= SAVE_EVERY:
            with open(OUT_PATH, "w", encoding="utf-8") as f:
                json.dump(preds, f, ensure_ascii=False)
            print("Autosaved:", len(preds))
            new_done = 0

        #
        buffer_q, buffer_e, buffer_ids = [], [], []

# flush remainder
if buffer_q:
    answers = answer_batch(buffer_q, buffer_e, max_new_tokens=MAX_NEW_TOKENS)
    for i, a in zip(buffer_ids, answers):
        preds[i] = a

with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(preds, f, ensure_ascii=False)

print("Saved:", OUT_PATH, "Num preds:", len(preds))


In [ ]:
!pip -q install ujson
!wget -q -O hotpot_evaluate_v1.py https://raw.githubusercontent.com/hotpotqa/hotpot/master/hotpot_evaluate_v1.py

In [ ]:
import json

DATA_PATH = "/content/drive/MyDrive/hotpot/hotpot_dev_fullwiki_v1.json"   # gold file
PRED_PATH = "/content/drive/MyDrive/hotpot/pred_dev_rag1.json"                # your answers
PRED_EVAL_PATH = "/content/drive/MyDrive/hotpot/pred_for_eval_rag1.json"      # new file

with open(PRED_PATH, "r", encoding="utf-8") as f:
    answers = json.load(f)

pred_for_eval = {
    "answer": answers,
    "sp": {qid: [] for qid in answers}   # empty supporting facts (baseline)
}

with open(PRED_EVAL_PATH, "w", encoding="utf-8") as f:
    json.dump(pred_for_eval, f, ensure_ascii=False)

print("Wrote:", PRED_EVAL_PATH)

In [ ]:
!python hotpot_evaluate_v1.py "/content/drive/MyDrive/hotpot/pred_for_eval_rag1.json" "/content/drive/MyDrive/hotpot/hotpot_dev_fullwiki_v1.json"